# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'
!rm -rf /kaggle/working/*

In [2]:
# Notebook-local constants and metadata
TASK_ID = 'task143'
MODEL_VERSION = 'task143-dynamic-exemplar-shape-recolor-30x30-static-graph-static-graph'
FAMILY = 'arc_static_non_tree_symbolic'
SUBTYPE = 'dynamic top-left exemplar shape recolors same-shape isolated objects to color 5; static graph without ScatterND/Shape/Range/Expand'
BUILDER = 'Dyn143StaticOnly'
EXPECTED_VISIBLE_PASS = True


In [3]:
# Dependencies
import importlib.util, subprocess, sys
required = {'onnx':'onnx', 'onnxruntime':'onnxruntime', 'torch':'torch', 'numpy':'numpy'}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
# legacy exporter is used, so onnxscript is not required
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 57.8 MB/s eta 0:00:00


In [4]:
# Shared non-tree ARC static ONNX modelling code for task143.
# This version keeps the Kaggle-compatible 30x30 signature and avoids dynamic-shape/scatter ops.
# In particular, the exported ONNX has no ScatterND, Shape, Range, Expand, Gather, ConstantOfShape,
# and none of the explicitly forbidden ops Loop/Scan/NonZero/Unique/Script/Function.
import json, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort

H = W = 30
CH = 10
FORBIDDEN_OPS = {'Loop','Scan','NonZero','Unique','Script','Function'}
STATIC_RISK_OPS = {'ScatterND','Shape','Range','Expand','Gather','ConstantOfShape'}
SIZE_LIMIT_BYTES = 1_400_000

class Dyn143StaticOnly(nn.Module):
    """Dynamic non-tree rule model for task143, exported as a static-shape/static-graph ONNX.

    Rule:
    1. Read the nonzero/non-5 occupancy pattern in the top-left 3x3 exemplar region.
    2. Find every isolated same-shape object in the 10x10 task region, regardless of its color.
    3. Recolor those matched object pixels to 5.
    4. Return a padded 30x30 one-hot tensor.

    Implementation note:
    The older Dyn143StaticOnly used slice assignment. PyTorch exported that as ScatterND plus
    Shape/Range/Expand scaffolding. Some Kaggle evaluators accept ONNXRuntime-valid models
    but score/reject models with those dynamic/scatter ops. This class uses only fixed
    slicing, convolution, arithmetic, concatenation, and fixed zero buffers.
    """
    def __init__(self):
        super().__init__()
        ring = torch.ones(1, 1, 5, 5)
        ring[:, :, 1:4, 1:4] = 0
        mask8 = torch.ones(1, 1, 8, 8)
        mask8[:, :, 0, 0] = 0
        self.register_buffer('ring', ring)
        self.register_buffer('mask8', mask8)
        self.register_buffer('z8_1', torch.zeros(1, 1, 8, 1))
        self.register_buffer('z8_2', torch.zeros(1, 1, 8, 2))
        self.register_buffer('z10_1', torch.zeros(1, 1, 1, 10))
        self.register_buffer('z10_2', torch.zeros(1, 1, 2, 10))
        self.register_buffer('z10_20', torch.zeros(1, 1, 10, 20))
        self.register_buffer('z20_30', torch.zeros(1, 1, 20, 30))

    def pad_to10(self, t, r: int, c: int):
        # t is [1,1,8,8]; place it at offset (r,c) inside a fixed [1,1,10,10] tensor.
        if c == 1:
            t = torch.cat([self.z8_1, t, self.z8_1], dim=3)
        elif c == 2:
            t = torch.cat([self.z8_2, t], dim=3)
        else:
            t = torch.cat([t, self.z8_2], dim=3)

        if r == 1:
            t = torch.cat([self.z10_1, t, self.z10_1], dim=2)
        elif r == 2:
            t = torch.cat([self.z10_2, t], dim=2)
        else:
            t = torch.cat([t, self.z10_2], dim=2)
        return t

    def pad10to30(self, t):
        # t is [1,1,10,10]; pad right and bottom with fixed zero buffers.
        t = torch.cat([t, self.z10_20], dim=3)
        t = torch.cat([t, self.z20_30], dim=2)
        return t

    def forward(self, x):
        # Static Kaggle input is 30x30, but task143 content is in the top-left 10x10 region.
        z = x[:, :, :10, :10]
        active = (z.sum(1, keepdim=True) > 0).float()

        # Dynamic reference shape: occupancy of top-left 3x3 non-background/non-target-color pixels.
        non0_non5 = (z.sum(1, keepdim=True) - z[:, 0:1] - z[:, 5:6]).clamp(0, 1)
        ref = non0_non5[:, :, :3, :3]
        refbits = [ref[:, :, r:r+1, c:c+1] for r in range(3) for c in range(3)]
        ref_area = sum(refbits)

        zero = z[:, 0:1] * 0.0
        add5 = zero
        used = zero

        # Check every possible 3x3 window in the 10x10 region for every nonzero/non-5 color.
        # Ring isolation prevents a 3x3 sub-window inside a larger same-color object from matching.
        for k in range(1, CH):
            if k == 5:
                continue
            m = z[:, k:k+1]
            bits = [m[:, :, r:r+8, c:c+8] for r in range(3) for c in range(3)]
            area = sum(bits)
            eq = area * 0.0 + 1.0
            for b, rb in zip(bits, refbits):
                eq = eq * (b * rb + (1 - b) * (1 - rb))
            ring_sum = F.conv2d(m, self.ring, padding=1)
            match = (area == ref_area).float() * eq * (ring_sum == 0).float() * (ref_area > 0).float() * self.mask8
            idx = 0
            for r in range(3):
                for c in range(3):
                    term = self.pad_to10(match * refbits[idx], r, c)
                    add5 = add5 + term
                    used = used + term
                    idx += 1

        add5 = add5.clamp(0, 1)
        used = used.clamp(0, 1)

        outs = []
        for k in range(CH):
            if k == 5:
                ch = (z[:, 5:6] + add5).clamp(0, 1)
            elif k == 0:
                ch = zero
            else:
                ch = z[:, k:k+1] * (1 - used)
            outs.append(self.pad10to30(ch))

        occupied = sum(outs[1:]).clamp(0, 1)
        active30 = self.pad10to30(active)
        outs[0] = (1 - occupied) * active30
        return torch.cat(outs, 1)

def make_model(task_id):
    assert task_id == 'task143', task_id
    return Dyn143StaticOnly()

def grid_to_tensor(grid):
    arr = np.zeros((1, CH, H, W), np.float32)
    for r, row in enumerate(grid[:H]):
        for c, v in enumerate(row[:W]):
            arr[0, int(v), r, c] = 1.0
    return arr

def expected_tensor(grid):
    return grid_to_tensor(grid)

def find_task_json(task_id):
    candidates = [
        Path.cwd() / f'{task_id}.json',
        Path('/kaggle/input/competitions/neurogolf-2026') / f'{task_id}.json',
        Path('/kaggle/input') / f'{task_id}.json',
        Path('/mnt/data') / f'{task_id}.json',
        Path('/mnt/data/generated_arc_notebooks_fixed') / f'{task_id}.json',
        Path('/mnt/data/generated_arc_notebooks') / f'{task_id}.json',
    ]
    for p in candidates:
        if p.exists():
            return p
    # fallback recursive search under Kaggle input; safe for small competition directory
    for root in [Path('/kaggle/input'), Path('/mnt/data')]:
        if root.exists():
            hits = list(root.rglob(f'{task_id}.json'))
            if hits:
                return hits[0]
    raise FileNotFoundError(f'{task_id}.json not found')

def load_task(task_id):
    p = find_task_json(task_id)
    with open(p, 'r') as f:
        return json.load(f), p

def export_onnx_model(task_id, model_path):
    model = make_model(task_id).eval()
    dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)
    torch.onnx.export(
        model, dummy, str(model_path),
        input_names=['input'], output_names=['output'],
        opset_version=13, dynamic_axes=None, do_constant_folding=True,
        dynamo=False,
    )
    m = onnx.load(str(model_path))
    # Keep IR compatible with older ONNX Runtime versions used by many Kaggle evaluators.
    m.ir_version = min(m.ir_version, 8)
    # Write static shape explicitly.
    for value_info, dims in [(m.graph.input[0], [1, CH, H, W]), (m.graph.output[0], [1, CH, H, W])]:
        for dim, value in zip(value_info.type.tensor_type.shape.dim, dims):
            dim.ClearField('dim_param')
            dim.dim_value = int(value)
    onnx.checker.check_model(m)
    onnx.save(m, str(model_path))
    return model_path

def _onnx_op_counts(model_path):
    on = onnx.load(str(model_path))
    ops = {}
    for node in on.graph.node:
        ops[node.op_type] = ops.get(node.op_type, 0) + 1
    return ops

def _onnx_signature(model_path):
    on = onnx.load(str(model_path))
    def dims(value_info):
        out = []
        for d in value_info.type.tensor_type.shape.dim:
            out.append(d.dim_value if d.dim_value else d.dim_param)
        return out
    return {
        'input': dims(on.graph.input[0]),
        'output': dims(on.graph.output[0]),
        'opset': [op.version for op in on.opset_import],
        'ir_version': on.ir_version,
    }

def make_ort_session(model_path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1
    so.inter_op_num_threads = 1
    return ort.InferenceSession(str(model_path), sess_options=so, providers=['CPUExecutionProvider'])

def validate_onnx_model(model_path, task, scope='visible'):
    sess = make_ort_session(model_path)
    splits = ['train','test'] if scope == 'visible' else ['train','test','arc-gen']
    right = total = wrongpix = 0
    first_wrong = None
    for split in splits:
        for idx, ex in enumerate(task.get(split, [])):
            if 'output' not in ex:
                continue
            pred = sess.run(['output'], {'input': grid_to_tensor(ex['input'])})[0]
            pred = (pred > 0.5).astype(np.float32)
            gold = expected_tensor(ex['output'])
            ok = np.array_equal(pred, gold)
            if ok:
                right += 1
            elif first_wrong is None:
                first_wrong = f'{split}:{idx}'
            wrongpix += int(np.sum(pred != gold))
            total += 1
    ops = _onnx_op_counts(model_path)
    return {
        'right': right,
        'total': total,
        'wrongpix': wrongpix,
        'first_wrong': first_wrong,
        'size': Path(model_path).stat().st_size,
        'under_1_4mb': Path(model_path).stat().st_size < SIZE_LIMIT_BYTES,
        'forbidden': sorted(FORBIDDEN_OPS & set(ops)),
        'static_risk_ops_present': sorted(STATIC_RISK_OPS & set(ops)),
        'signature': _onnx_signature(model_path),
        'ops': ops,
    }

def validate_arcgen_split(model_path, task, seed=0, ratio=0.30):
    arc = task.get('arc-gen', [])
    idx = list(range(len(arc)))
    random.Random(seed).shuffle(idx)
    n_test = max(1, int(round(len(idx) * ratio))) if idx else 0
    test_idx = set(idx[:n_test])
    sess = make_ort_session(model_path)
    fit_right = fit_total = test_right = test_total = 0
    for i, ex in enumerate(arc):
        pred = sess.run(['output'], {'input': grid_to_tensor(ex['input'])})[0]
        pred = (pred > 0.5).astype(np.float32)
        gold = expected_tensor(ex['output'])
        ok = int(np.array_equal(pred, gold))
        if i in test_idx:
            test_right += ok; test_total += 1
        else:
            fit_right += ok; fit_total += 1
    return {
        'seed': seed,
        'ratio': ratio,
        'fit_right': fit_right,
        'fit_total': fit_total,
        'test_right': test_right,
        'test_total': test_total,
    }


In [5]:
from pathlib import Path
import json
ROOT = Path.cwd()
OUT_DIR = ROOT / 'generated_models'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUT_DIR / f'{TASK_ID}.onnx'
MANIFEST_PATH = OUT_DIR / f'{TASK_ID}_manifest.json'

In [6]:
# Load task JSON. This cell includes a lightweight fallback loader so it works even if
# the large shared modelling cell above was not run before this cell.
from pathlib import Path
import json

def _candidate_task_roots():
    roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path('/mnt/data'),
        Path('/mnt/data/generated_arc_notebooks_fixed'),
        Path('/mnt/data/generated_arc_notebooks/extracted'),
    ]
    seen = []
    for r in roots:
        if r not in seen:
            seen.append(r)
    return seen

if 'find_task_json' not in globals():
    def find_task_json(task_id):
        names = [f'{task_id}.json']
        for root in _candidate_task_roots():
            for name in names:
                p = root / name
                if p.exists():
                    return p
        raise FileNotFoundError(
            f'Cannot find {task_id}.json. Put the task JSON next to the notebook, '
            f'in the parent folder, or in /mnt/data.'
        )

if 'load_task' not in globals():
    def load_task(task_id):
        p = find_task_json(task_id)
        with open(p) as f:
            return json.load(f), p

task, task_path = load_task(TASK_ID)
print('task path:', task_path)
print('train examples:', len(task.get('train', [])))
print('test examples:', len(task.get('test', [])))
print('arc-gen examples:', len(task.get('arc-gen', [])))


task path: /kaggle/input/competitions/neurogolf-2026/task143.json
train examples: 3
test examples: 1
arc-gen examples: 262


In [7]:
# Small inspection: shape and color counts for the first training pair.
from collections import Counter
first = task['train'][0]
print('input shape:', (len(first['input']), len(first['input'][0])))
print('output shape:', (len(first['output']), len(first['output'][0])))
print('input colors:', Counter(v for row in first['input'] for v in row))
print('output colors:', Counter(v for row in first['output'] for v in row))

input shape: (10, 10)
output shape: (10, 10)
input colors: Counter({0: 69, 7: 9, 5: 7, 1: 5, 6: 5, 8: 5})
output colors: Counter({0: 69, 5: 12, 7: 9, 1: 5, 8: 5})


In [8]:
# Export plan for this task.
plan = {
    'task_id': TASK_ID,
    'builder': BUILDER,
    'family': FAMILY,
    'subtype': SUBTYPE,
    'static_input_shape': [1, 10, 30, 30],
    'static_output_shape': [1, 10, 30, 30],
    'acceptance': [
        'visible train/test exact pass',
        'full arc-gen exact pass',
        '30% arc-gen split exact pass',
        'ONNX < 1.4MB',
        'no Loop/Scan/NonZero/Unique/Script/Function',
        'non-tree, no visible-template lookup',
        'writes submission.zip',
    ],
}
print(json.dumps(plan, indent=2))


{
  "task_id": "task143",
  "builder": "Dyn143StaticOnly",
  "family": "arc_static_non_tree_symbolic",
  "subtype": "dynamic top-left exemplar shape recolors same-shape isolated objects to color 5; static graph without ScatterND/Shape/Range/Expand",
  "static_input_shape": [
    1,
    10,
    30,
    30
  ],
  "static_output_shape": [
    1,
    10,
    30,
    30
  ],
  "acceptance": [
    "visible train/test exact pass",
    "full arc-gen exact pass",
    "30% arc-gen split exact pass",
    "ONNX < 1.4MB",
    "no Loop/Scan/NonZero/Unique/Script/Function",
    "non-tree, no visible-template lookup",
    "writes submission.zip"
  ]
}


In [9]:
# Task-specific model construction wrapper.
def build_current_model():
    return export_onnx_model(TASK_ID, MODEL_PATH)

def validate_current_model(scope='visible'):
    return validate_onnx_model(MODEL_PATH, task, scope=scope)

In [10]:
# Build ONNX model and enforce structural competition constraints.
build_current_model()
validation_report = validate_current_model('visible')
print(json.dumps(validation_report, indent=2)[:5000])
assert validation_report['signature']['input'] == [1, 10, 30, 30], validation_report['signature']
assert validation_report['signature']['output'] == [1, 10, 30, 30], validation_report['signature']
assert validation_report['under_1_4mb'], validation_report
assert not validation_report['forbidden'], validation_report
assert validation_report['right'] == validation_report['total'], validation_report

assert not validation_report.get('static_risk_ops_present'), validation_report.get('static_risk_ops_present')


/tmp/ipykernel_16/509631010.py:169: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


{
  "right": 4,
  "total": 4,
  "wrongpix": 0,
  "first_wrong": null,
  "size": 138051,
  "under_1_4mb": true,
  "forbidden": [],
  "static_risk_ops_present": [],
  "signature": {
    "input": [
      1,
      10,
      30,
      30
    ],
    "output": [
      1,
      10,
      30,
      30
    ],
    "opset": [
      13
    ],
    "ir_version": 7
  },
  "ops": {
    "Constant": 613,
    "Slice": 122,
    "ReduceSum": 1,
    "Greater": 2,
    "Cast": 18,
    "Sub": 85,
    "Clip": 4,
    "Add": 239,
    "Mul": 338,
    "Conv": 8,
    "Equal": 16,
    "Concat": 165
  }
}


In [11]:
# Strict broader arc-gen check: must pass full arc-gen and a fixed 30% arc-gen test split.
optional_all_report = validate_current_model('all')
arcgen_split_report = validate_arcgen_split(MODEL_PATH, task, seed=0, ratio=0.30)
print('full validation:', json.dumps(optional_all_report, indent=2)[:5000])
print('arc-gen split:', json.dumps(arcgen_split_report, indent=2))
assert optional_all_report['right'] == optional_all_report['total'], optional_all_report
assert arcgen_split_report['test_right'] == arcgen_split_report['test_total'], arcgen_split_report

assert not optional_all_report.get('static_risk_ops_present'), optional_all_report.get('static_risk_ops_present')


full validation: {
  "right": 266,
  "total": 266,
  "wrongpix": 0,
  "first_wrong": null,
  "size": 138051,
  "under_1_4mb": true,
  "forbidden": [],
  "static_risk_ops_present": [],
  "signature": {
    "input": [
      1,
      10,
      30,
      30
    ],
    "output": [
      1,
      10,
      30,
      30
    ],
    "opset": [
      13
    ],
    "ir_version": 7
  },
  "ops": {
    "Constant": 613,
    "Slice": 122,
    "ReduceSum": 1,
    "Greater": 2,
    "Cast": 18,
    "Sub": 85,
    "Clip": 4,
    "Add": 239,
    "Mul": 338,
    "Conv": 8,
    "Equal": 16,
    "Concat": 165
  }
}
arc-gen split: {
  "seed": 0,
  "ratio": 0.3,
  "fit_right": 183,
  "fit_total": 183,
  "test_right": 79,
  "test_total": 79
}


In [12]:
# Model/version manifest.
run_manifest = {
    'task_id': TASK_ID,
    'model_version': MODEL_VERSION,
    'family': FAMILY,
    'subtype': SUBTYPE,
    'builder': BUILDER,
    'uses_tree_methods': False,
    'uses_visible_template_lookup': False,
    'static_input_shape': [1, 10, 30, 30],
    'static_output_shape': [1, 10, 30, 30],
    'visible_validation': validation_report,
    'arc_gen_validation': optional_all_report,
    'arc_gen_split_validation': arcgen_split_report,
}
print(json.dumps(run_manifest, indent=2)[:5000])


{
  "task_id": "task143",
  "model_version": "task143-dynamic-exemplar-shape-recolor-30x30-static-graph-static-graph",
  "family": "arc_static_non_tree_symbolic",
  "subtype": "dynamic top-left exemplar shape recolors same-shape isolated objects to color 5; static graph without ScatterND/Shape/Range/Expand",
  "builder": "Dyn143StaticOnly",
  "uses_tree_methods": false,
  "uses_visible_template_lookup": false,
  "static_input_shape": [
    1,
    10,
    30,
    30
  ],
  "static_output_shape": [
    1,
    10,
    30,
    30
  ],
  "visible_validation": {
    "right": 4,
    "total": 4,
    "wrongpix": 0,
    "first_wrong": null,
    "size": 138051,
    "under_1_4mb": true,
    "forbidden": [],
    "static_risk_ops_present": [],
    "signature": {
      "input": [
        1,
        10,
        30,
        30
      ],
      "output": [
        1,
        10,
        30,
        30
      ],
      "opset": [
        13
      ],
      "ir_version": 7
    },
    "ops": {
      "Constant":

In [13]:
# Architecture report.
model = onnx.load(str(MODEL_PATH))
op_counts = {}
for node in model.graph.node:
    op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
print('onnx_size_bytes:', MODEL_PATH.stat().st_size)
print('forbidden_ops:', sorted(FORBIDDEN_OPS & set(op_counts)))
print('op_counts:', op_counts)

onnx_size_bytes: 138051
forbidden_ops: []
op_counts: {'Constant': 613, 'Slice': 122, 'ReduceSum': 1, 'Greater': 2, 'Cast': 18, 'Sub': 85, 'Clip': 4, 'Add': 239, 'Mul': 338, 'Conv': 8, 'Equal': 16, 'Concat': 165}


In [14]:
# Persist metadata next to ONNX.
with open(MANIFEST_PATH, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('manifest:', MANIFEST_PATH)

manifest: /kaggle/working/generated_models/task143_manifest.json


In [15]:
# Package single-task submission zip.
zip_path = ROOT / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_PATH, MODEL_PATH.name)
print('submission zip:', zip_path)

submission zip: /kaggle/working/submission.zip


In [16]:
# Persist run metadata next to generated models.
summary_path = OUT_DIR / f'{TASK_ID}_verification_summary.json'
with open(summary_path, 'w') as f:
    json.dump({'visible': validation_report, 'all': optional_all_report, 'arcgen_split': arcgen_split_report}, f, indent=2)
print('summary:', summary_path)


summary: /kaggle/working/generated_models/task143_verification_summary.json
